In [1]:
from pyspark.sql import SparkSession
import pathling

In [5]:
spark = (
    SparkSession.builder
    .appName("snomed-dictionary")
    .master("local[*]")
    .config("spark.hadoop.hadoop.security.authentication", "simple")
    .config("spark.hadoop.hadoop.security.authorization", "false")
    .config("spark.hadoop.hadoop.job.ugi", "localuser,localgroup")
    .getOrCreate()
)


Py4JJavaError: An error occurred while calling None.org.apache.spark.api.java.JavaSparkContext.
: java.lang.UnsupportedOperationException: getSubject is not supported
	at java.base/javax.security.auth.Subject.getSubject(Subject.java:277)
	at org.apache.hadoop.security.UserGroupInformation.getCurrentUser(UserGroupInformation.java:588)
	at org.apache.spark.util.Utils$.$anonfun$getCurrentUserName$1(Utils.scala:2446)
	at scala.Option.getOrElse(Option.scala:201)
	at org.apache.spark.util.Utils$.getCurrentUserName(Utils.scala:2446)
	at org.apache.spark.SparkContext.<init>(SparkContext.scala:339)
	at org.apache.spark.api.java.JavaSparkContext.<init>(JavaSparkContext.scala:59)
	at java.base/jdk.internal.reflect.DirectConstructorHandleAccessor.newInstance(DirectConstructorHandleAccessor.java:62)
	at java.base/java.lang.reflect.Constructor.newInstanceWithCaller(Constructor.java:499)
	at java.base/java.lang.reflect.Constructor.newInstance(Constructor.java:483)
	at py4j.reflection.Metho

In [ ]:
diagnoses_path = "D:/Python Projects/Hospital readmission risk/SQLs/raw_data_for_dictionaries/unique_diagnoses_test.csv"

df_diag = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(diagnoses_path)
)

df_diag.show(5)


In [1]:
from scttsrapy.concepts import get_concept


In [2]:
def get_snomed_description(concept_id: str) -> str:
    """
    Return a human-readable description for a SNOMED CT concept ID.
    Prefers the preferred term (pt.term); falls back to FSN if needed.
    """
    response = get_concept(concept_id)

    if not response.get("success"):
        raise ValueError(f"Lookup failed for concept {concept_id}: {response}")

    content = response.get("content") or {}

    # Preferred term first
    pt = content.get("pt")
    if pt and pt.get("term"):
        return pt["term"]

    # Fallback to fully specified name
    fsn = content.get("fsn")
    if fsn and fsn.get("term"):
        return fsn["term"]

    # Last resort: just return the ID
    return f"SNOMED concept {concept_id}"

In [3]:
concept_id = "10509002"  # replace with your SNOMED CT code
print(get_snomed_description(concept_id))

ConnectTimeout: HTTPSConnectionPool(host='snowstorm.ihtsdotools.org', port=443): Max retries exceeded with url: /snowstorm/snomed-ct/MAIN/concepts/10509002 (Caused by ConnectTimeoutError(<HTTPSConnection(host='snowstorm.ihtsdotools.org', port=443) at 0x295ce195c10>, 'Connection to snowstorm.ihtsdotools.org timed out. (connect timeout=None)'))